# NumPy Advanced — Zero-Copy Pipelines, Vectorisation & Performance

**Module 2.2 | Edge AI Engineering Bootcamp**

This notebook covers advanced NumPy techniques that are essential for building **high-performance** Edge AI pipelines on resource-constrained devices like a Raspberry Pi:

1. `np.frombuffer()` — zero-copy camera frame handling
2. `np.ascontiguousarray()` — fixing memory layout issues
3. Vectorised operations vs Python loops — the 100x speed difference
4. Advanced indexing patterns
5. Memory layout: C-order vs Fortran-order
6. `np.einsum()` — Einstein notation for complex operations
7. Performance benchmarking and optimization techniques

In [1]:
import numpy as np
import time
import sys

---
## 1. np.frombuffer() — Zero-Copy Camera Pipelines

When working with cameras, copying image data unnecessarily wastes CPU time and memory bandwidth — and on a Pi, bandwidth is precious. `np.frombuffer()` lets you **avoid copies entirely**.

Many camera libraries hand you raw image data as a **buffer** (a block of bytes) rather than a ready-made NumPy array. `np.frombuffer()` wraps that memory directly as an array **without copying it**.

### Why this matters

Copying a full-resolution image frame 30 times a second (30 FPS) wastes real CPU cycles that your AI model could be using instead. `frombuffer` reads the data in place.

### Important caveat

The array from `frombuffer` is **read-only by default** — because it doesn't own the memory, NumPy won't let you casually modify data that something else (like the camera driver) might still be using.

```
Raw buffer in memory:  [byte][byte][byte][byte]...
np.frombuffer() doesn't copy anything —
it just tells NumPy: "treat this existing
memory as an array, using this dtype and shape"
```

### `np.frombuffer` in Action: Turning Raw Bytes into an Array

Let's see exactly how `frombuffer` works — it takes a block of raw bytes and interprets them as a NumPy array **without copying any data**.

### The Function Signature

```python
np.frombuffer(buffer, dtype=np.uint8)
```

- `buffer`: any object that exposes raw bytes (a `bytes` object, a `bytearray`, a memoryview, etc.)
- `dtype`: how to interpret each element (e.g., `uint8` = 1 byte per element)

### What Happens Step by Step

```
1. Camera gives you: 921,600 raw bytes (480 × 640 × 3 = 921,600)
2. np.frombuffer(raw_bytes, dtype=np.uint8)
   → Creates a 1D array of shape (921,600,) pointing to the SAME memory
   → No data copied — the array is just a "lens" over the buffer
3. .reshape((480, 640, 3))
   → Changes the shape to 3D — still no copy (just changes strides)
4. Now you have a (480, 640, 3) image array with zero bytes duplicated
```

### Why `reshape` After `frombuffer` Is Free

`frombuffer` returns a 1D array. `reshape` just changes how NumPy indexes the same flat memory — it adjusts strides, no data moves. So the full pipeline `frombuffer().reshape()` is **completely zero-copy**.

### Memory Layout After `frombuffer`

```
Raw buffer in memory:  [B0][B1][B2][B3][B4][B5]...
                        ↑
frombuffer points here  |
                        B0 = pixel[0,0,0] (Red)
                        B1 = pixel[0,0,1] (Green)
                        B2 = pixel[0,0,2] (Blue)
                        B3 = pixel[0,1,0] (Red of next pixel)
                        ...
```

In [2]:
# --- np.frombuffer(): turning raw bytes into an array without copying ---

# Simulate raw bytes from a camera (480 * 640 * 3 = 921,600 bytes)
raw_bytes = bytes(np.random.randint(0, 256, 921600, dtype=np.uint8))
print(f"Raw buffer size: {len(raw_bytes):,} bytes")
print(f"Raw buffer type: {type(raw_bytes)}")

# Interpret those exact same bytes as a NumPy array, NO copying:
frame = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((480, 640, 3))
print(f"\nFrame shape: {frame.shape}")
print(f"Frame dtype: {frame.dtype}")
print(f"Frame data pointer: {frame.ctypes.data}")  # Same memory location!
print(f"Raw buffer id: {id(raw_bytes)}")

# Verify no copy was made — frombuffer shares memory with the original
# (We can't use shares_memory directly with bytes, but we can check that
# the array is a view of the buffer's data)

Raw buffer size: 921,600 bytes
Raw buffer type: <class 'bytes'>

Frame shape: (480, 640, 3)
Frame dtype: uint8
Frame data pointer: 50185895968
Raw buffer id: 50185895936


### The Read-Only Nature of `frombuffer`

The array created by `np.frombuffer()` is **read-only** by default. This is a safety mechanism, not a bug.

### Why It's Read-Only

The array doesn't own the memory it points to — the camera driver (or whatever created the buffer) does. If NumPy let you modify it, you could corrupt data that another part of the program is still using:

```
Camera driver:  "Here's a buffer with frame data"
np.frombuffer:  "I'll wrap this as an array (read-only)"
You:            "I want to modify pixel [0,0]"
NumPy:          "No — you don't own this memory. Make a copy first."
```

### How to Get a Writable Array

```python
writable = np.frombuffer(raw_bytes, dtype=np.uint8).copy()
writable[0] = 255  # Now it works — .copy() gives you your own memory
```

### Checking Writability

```python
arr.flags.writeable   # True = you can modify, False = read-only
```

### When Read-Only Is Fine

If you're only **reading** the frame (e.g., passing it to an AI model for inference), read-only is perfect — no copy needed, zero extra memory. Only call `.copy()` when you actually need to modify the data.

In [3]:
# --- The read-only nature of frombuffer ---

raw_bytes = bytes(np.random.randint(0, 256, 12, dtype=np.uint8))
frame = np.frombuffer(raw_bytes, dtype=np.uint8)

print("Array from frombuffer:", frame)
print("Is writable?", frame.flags.writeable)  # False

# Attempting to modify raises an error
try:
    frame[0] = 255
except ValueError as e:
    print(f"\nError when trying to modify: {e}")
    print("→ The array is read-only because it doesn't own the memory!")

# If you genuinely need to modify it, make an explicit copy:
writable_frame = np.frombuffer(raw_bytes, dtype=np.uint8).copy()
writable_frame[0] = 255
print(f"\nAfter .copy() and modification: {writable_frame}")
print("Is writable?", writable_frame.flags.writeable)  # True

Array from frombuffer: [ 94 200  34 248  41 144  84 139 159 160 202  11]
Is writable? False

Error when trying to modify: assignment destination is read-only
→ The array is read-only because it doesn't own the memory!

After .copy() and modification: [255 200  34 248  41 144  84 139 159 160 202  11]
Is writable? True


### Simulating a Real Camera Pipeline

Let's put `frombuffer` in context with a simulated camera that returns raw bytes — exactly like a real PiCamera or OpenCV `VideoCapture` would.

### The Real-World Pipeline

```
Camera driver → raw bytes (buffer) → np.frombuffer() → reshape → image array
                                                                      ↓
                                                              .copy() if needed
                                                                      ↓
                                                           .astype(float32) / 255
                                                                      ↓
                                                              AI model input
```

### Key Decisions in the Pipeline

| Step | Copy or not? | Why? |
|---|---|---|
| `np.frombuffer(raw_bytes)` | **No copy** | Wraps existing memory |
| `.reshape((480, 640, 3))` | **No copy** | Just changes strides |
| `.copy()` | **Copy** | Need writable array for processing |
| `.astype(np.float32)` | **Copy** | Different dtype = new memory |
| `/ 255.0` | **Copy** | Arithmetic creates new array |

### The Memory Budget

On a Raspberry Pi with 1GB RAM, processing a 480×640×3 frame:
- Raw bytes: ~0.9 MB (from camera)
- `frombuffer` array: 0 MB extra (same memory)
- Copy for processing: +0.9 MB
- Float32 conversion: +3.5 MB
- Total per frame: ~5.3 MB

At 30 FPS, that's ~160 MB/s of memory traffic. Avoiding unnecessary copies isn't just about speed — it's about not running out of memory.

In [4]:
# --- Simulating a real camera pipeline with frombuffer ---

# In real code, this would be from a camera library like:
#   raw_bytes = camera.get_raw_frame()  # e.g., from PiCamera, OpenCV VideoCapture, etc.

# Simulate a camera that provides raw bytes
class SimulatedCamera:
    """Simulates a camera that returns raw byte buffers."""
    def __init__(self, width=640, height=480, channels=3):
        self.width = width
        self.height = height
        self.channels = channels
    
    def get_raw_frame(self):
        # Returns raw bytes (like a real camera driver would)
        return np.random.randint(0, 256, 
            self.height * self.width * self.channels, dtype=np.uint8).tobytes()

camera = SimulatedCamera()

# Zero-copy pipeline: interpret raw bytes as an image array
raw_bytes = camera.get_raw_frame()
frame = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((480, 640, 3))

print(f"Camera frame: shape={frame.shape}, dtype={frame.dtype}")
print(f"Memory used by raw_bytes: {sys.getsizeof(raw_bytes):,} bytes")
print(f"No extra copy — frombuffer uses the same memory!")

# If you need to process the frame (which requires writing), copy first
processed = frame.copy()  # Only now do we allocate new memory
processed = processed.astype(np.float32) / 255.0  # Normalize for AI model
print(f"\nProcessed frame: shape={processed.shape}, dtype={processed.dtype}")
print(f"Value range: [{processed.min():.3f}, {processed.max():.3f}]")

Camera frame: shape=(480, 640, 3), dtype=uint8
Memory used by raw_bytes: 921,633 bytes
No extra copy — frombuffer uses the same memory!

Processed frame: shape=(480, 640, 3), dtype=float32
Value range: [0.000, 1.000]


---
## 2. np.ascontiguousarray() — Fixing Memory Layout Issues

Some operations (like slicing with a step, or transposing) leave your array's data **scattered in memory** in a non-standard order, even though it still behaves correctly logically. This can slow down or even **break** libraries (like OpenCV, or AI inference engines) that expect memory to be laid out simply and predictably.

### When to use it

- After a **transpose** (swapping axes)
- After **flipping with a step** (e.g., `img[::-1]`)
- After **fancy indexing** that returns non-contiguous data
- Before passing an array to **OpenCV, TensorFlow Lite, or other C libraries**

### Simple guideline

- If you just sliced normally (`img[100:300, 200:500]`), you're almost always still contiguous and fine.
- If you **transposed, flipped with a step, or did fancy indexing**, check `.flags['C_CONTIGUOUS']` before handing the array to another library.

### Checking and Fixing Contiguity

Every NumPy array has flags that tell you how it's laid out in memory. The most important one is `C_CONTIGUOUS`.

### The Key Flag

```python
arr.flags['C_CONTIGUOUS']   # True if data is stored row-by-row (C-order)
arr.flags['F_CONTIGUOUS']   # True if data is stored column-by-column (Fortran-order)
```

### What Transpose Does to Memory Layout

When you transpose an image from `(480, 640, 3)` to `(640, 480, 3)`:
- The **shape** changes — NumPy now indexes it as width-first
- The **memory** doesn't move — same bytes, same order
- The **strides** swap — NumPy reads the same data in a different order
- `C_CONTIGUOUS` becomes **False** — the data is no longer in the order that matches the new shape

```
Before transpose (480, 640, 3):          After transpose (640, 480, 3):
Memory: [row0][row1][row2]...            Memory: [row0][row1][row2]...  (unchanged!)
        ↑ contiguous, matches shape              ↑ NOT contiguous for new shape
```

### The Fix: `np.ascontiguousarray()`

```python
fixed = np.ascontiguousarray(transposed)
# Creates a new array with data rearranged to match the new shape's order
# C_CONTIGUOUS is now True
```

This physically rearranges the data in memory so it's stored in the order that matches the current shape. It's a copy operation, so use it only when needed.

In [5]:
# --- Checking and fixing contiguity ---

# Create a standard image array (C-contiguous by default)
img = np.zeros((480, 640, 3), dtype=np.uint8)
print("Original array:")
print(f"  C_CONTIGUOUS: {img.flags['C_CONTIGUOUS']}")  # True
print(f"  F_CONTIGUOUS: {img.flags['F_CONTIGUOUS']}")  # False

# Transpose: swap height and width — this creates a NON-contiguous array
transposed = img.transpose(1, 0, 2)  # (640, 480, 3) instead of (480, 640, 3)
print(f"\nAfter transpose(1, 0, 2):")
print(f"  Shape: {transposed.shape}")
print(f"  C_CONTIGUOUS: {transposed.flags['C_CONTIGUOUS']}")  # False!
print(f"  → Data is scattered in memory, even though it works logically")

# Fix it with np.ascontiguousarray()
fixed = np.ascontiguousarray(transposed)
print(f"\nAfter np.ascontiguousarray():")
print(f"  C_CONTIGUOUS: {fixed.flags['C_CONTIGUOUS']}")  # True
print(f"  → Data is now properly laid out in memory")

Original array:
  C_CONTIGUOUS: True
  F_CONTIGUOUS: False

After transpose(1, 0, 2):
  Shape: (640, 480, 3)
  C_CONTIGUOUS: False
  → Data is scattered in memory, even though it works logically

After np.ascontiguousarray():
  C_CONTIGUOUS: True
  → Data is now properly laid out in memory


### More Non-Contiguous Examples: Which Operations Break Contiguity?

Let's systematically check which common operations preserve contiguity and which break it.

### The Quick Reference

| Operation | Contiguous after? | Why? |
|---|---|---|
| Normal slice `img[100:300, 200:500]` | ✅ Yes | Sub-region of contiguous data — still in order |
| Step slice `img[::2, ::2]` | ❌ No | Skips elements — gaps in memory |
| Flip `img[::-1]` | ❌ No | Negative stride — reads backward |
| Transpose `img.T` | ❌ No | Swaps axes — different traversal order |
| Fancy indexing `img[[0, 2, 5]]` | ❌ No | Arbitrary selection — no regular pattern |

### Visual: Why Step Slicing Breaks Contiguity

```
Original memory (contiguous):
[0] [1] [2] [3] [4] [5] [6] [7] [8] [9]
 ↑               ↑               ↑
 img[::3] picks: 0, 3, 6 — gaps between selected elements!

Step-sliced view reads: [0] ___ ___ [3] ___ ___ [6] ___ ___ [9]
                         ↑              ↑              ↑
                    Not next to each other → NOT contiguous
```

### The Fix Is Always the Same

```python
fixed = np.ascontiguousarray(non_contiguous_array)
```

This creates a new array with the same data, laid out contiguously in memory. It does cost one copy, so only use it when needed (e.g., before passing to an external library).

In [6]:
# --- More non-contiguous examples ---

img = np.zeros((480, 640, 3), dtype=np.uint8)

# Example 1: Normal slice — still contiguous (just a sub-view)
crop = img[100:300, 200:500]
print(f"Normal slice [100:300, 200:500]: C_CONTIGUOUS = {crop.flags['C_CONTIGUOUS']}")
# True — normal slicing keeps contiguity

# Example 2: Step slice — NON-contiguous
strided = img[::2, ::2]  # Every other pixel
print(f"Step slice [::2, ::2]: C_CONTIGUOUS = {strided.flags['C_CONTIGUOUS']}")
# False — skipping elements breaks contiguity

# Example 3: Flip with step — NON-contiguous
flipped = img[::-1]
print(f"Flip [::-1]: C_CONTIGUOUS = {flipped.flags['C_CONTIGUOUS']}")
# False — reversed stride breaks contiguity

# Example 4: Transpose — NON-contiguous
transposed = img.T
print(f"Transpose (.T): C_CONTIGUOUS = {transposed.flags['C_CONTIGUOUS']}")
# False — swapped axes break contiguity

# Fix all of them with ascontiguousarray
print(f"\nAfter ascontiguousarray:")
print(f"  strided: {np.ascontiguousarray(strided).flags['C_CONTIGUOUS']}")
print(f"  flipped: {np.ascontiguousarray(flipped).flags['C_CONTIGUOUS']}")
print(f"  transposed: {np.ascontiguousarray(transposed).flags['C_CONTIGUOUS']}")

Normal slice [100:300, 200:500]: C_CONTIGUOUS = False
Step slice [::2, ::2]: C_CONTIGUOUS = False
Flip [::-1]: C_CONTIGUOUS = False
Transpose (.T): C_CONTIGUOUS = False

After ascontiguousarray:
  strided: True
  flipped: True
  transposed: True


### Real-World Scenario: Passing Arrays to External Libraries

Edge AI pipelines often pass NumPy arrays to **C-based libraries** like OpenCV, TensorFlow Lite, or ONNX Runtime. These libraries access memory directly through C pointers — they expect data to be **contiguous in memory** (C-order).

### What Can Go Wrong

If you pass a non-contiguous array to a C library, one of three things happens:

| Outcome | When it happens | Severity |
|---|---|---|
| **Error** | Library checks contiguity and rejects the array | Annoying but safe |
| **Wrong results** | Library reads memory assuming C-order, gets scrambled data | **Silent data corruption** |
| **Slow performance** | Library internally copies the array to make it contiguous | Wasted time + memory |

### The Safe Pattern

```python
def safe_for_external_lib(arr):
    if not arr.flags['C_CONTIGUOUS']:
        return np.ascontiguousarray(arr)
    return arr
```

Always wrap arrays with this check before passing to OpenCV, TFLite, or any C library. If the array is already contiguous, it's a no-op (returns the same array, no copy). If not, it fixes it.

### When You're Most at Risk

- After `img.transpose()` or `img.T` — axes swapped
- After `img[::-1]` — reversed (negative stride)
- After `img[::2, ::2]` — stepped (skipped elements)
- After fancy indexing `img[[0, 2, 5]]` — non-contiguous selection

In [7]:
# --- Real-world scenario: passing arrays to external libraries ---

# Simulate what happens when you pass a non-contiguous array to a C library
# (like OpenCV or TensorFlow Lite)

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# You transpose for some reason (e.g., rotating the image)
rotated = img.transpose(1, 0, 2)  # Now shape is (640, 480, 3)

# Many C libraries expect C-contiguous arrays and will either:
# 1. Raise an error
# 2. Silently produce wrong results
# 3. Be much slower (they have to copy internally)

# Safe approach: always call ascontiguousarray before passing to external libs
safe_rotated = np.ascontiguousarray(rotated)
print(f"Safe to pass to OpenCV/TFLite: C_CONTIGUOUS = {safe_rotated.flags['C_CONTIGUOUS']}")

# Check if an array needs fixing before passing to a library
def safe_for_external_lib(arr):
    """Ensure array is C-contiguous before passing to external C libraries."""
    if not arr.flags['C_CONTIGUOUS']:
        print(f"  WARNING: Array was not contiguous, fixing with ascontiguousarray()")
        return np.ascontiguousarray(arr)
    return arr

print("\nChecking rotated array:")
safe1 = safe_for_external_lib(rotated)  # Will fix it
print("\nChecking original array:")
safe2 = safe_for_external_lib(img)  # Already fine, no fix needed

Safe to pass to OpenCV/TFLite: C_CONTIGUOUS = True

Checking rotated array:

Checking original array:


---
## 3. Vectorised Operations vs Python Loops — The 100x Speed Difference

This is the **single most important performance lesson** in this entire module.

### The Golden Rule of NumPy (worth memorizing):

> If you're writing a for loop over array elements to do math, **stop** — there's almost always a vectorised NumPy way to do the same thing, and it will be dramatically faster.

### Why the difference is so massive

| Python loop | Vectorised NumPy |
|---|---|
| For each of 921,600 pixels: | One single instruction tells the CPU: |
| - Python interpreter overhead for EVERY element | "divide this ENTIRE block of memory by 255, |
| - no use of NEON SIMD | using NEON SIMD, all at once" |
| - constant type-checking | A handful of highly optimized, compiled C/assembly instructions |
| ~921,600 separate, slow Python operations | |

NumPy's internal operations are written in **C** and compiled to use **hardware acceleration like NEON SIMD** directly. A Python for loop pays the overhead cost of the Python interpreter for every single element, one at a time.

In [8]:
# --- Benchmark: Python loops vs vectorised NumPy ---

# Create a realistic image (VGA resolution, 3 channels)
img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# THE SLOW WAY — a Python for loop (don't do this!)
start = time.time()
normalized_loop = np.zeros(img.shape, dtype=np.float32)
for y in range(img.shape[0]):
    for x in range(img.shape[1]):
        for c in range(img.shape[2]):
            normalized_loop[y, x, c] = img[y, x, c] / 255.0
loop_time = time.time() - start
print(f"Loop version:      {loop_time:.4f} seconds")

# THE FAST WAY — vectorised NumPy
start = time.time()
normalized_vec = img.astype(np.float32) / 255.0
vec_time = time.time() - start
print(f"Vectorised version: {vec_time:.4f} seconds")

speedup = loop_time / vec_time
print(f"\n→ Vectorised is {speedup:.1f}x faster!")
print(f"→ On a Raspberry Pi, this difference decides whether your project runs in real-time or not")

# Verify both produce the same result
print(f"\nResults match? {np.allclose(normalized_loop, normalized_vec)}")

Loop version:      0.8574 seconds
Vectorised version: 0.0008 seconds

→ Vectorised is 1013.4x faster!
→ On a Raspberry Pi, this difference decides whether your project runs in real-time or not

Results match? True


### Common Vectorised Operations for Edge AI

Here are the most frequently used vectorised patterns in image processing pipelines. Each replaces what would otherwise be a multi-level nested loop.

### The Pattern Library

| Task | Loop version (don't do) | Vectorised version (do this) |
|---|---|---|
| Adjust brightness | `for y, x, c: img[y,x,c] += 30` | `np.clip(img.astype(np.int16) + 30, 0, 255).astype(np.uint8)` |
| Threshold | `for y, x: if img[y,x] > 128: ...` | `mask = img > 128` |
| Per-channel mean | `for c: for y, x: total += img[y,x,c]` | `img.mean(axis=(0, 1))` |
| Count bright pixels | `for y, x: if img[y,x] > 200: count++` | `np.sum(img > 200)` |
| Normalize to [0,1] | `for y, x, c: img[y,x,c] /= 255` | `img.astype(np.float32) / 255.0` |
| Standardize per channel | Multiple nested loops | `(img_f32 - mean) / std` (broadcasting!) |

### Why Vectorisation Is More Than Just Speed

- **Readability**: one line vs 5+ lines of nested loops
- **Correctness**: no off-by-one errors in loop bounds
- **Maintainability**: easier to modify and debug
- **Portability**: NumPy's C backend uses the same optimized code on every platform

In [9]:
# --- More vectorisation examples for common Edge AI tasks ---

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# 1. Adjust brightness of entire image at once (safe, no wraparound)
brighter = np.clip(img.astype(np.int16) + 30, 0, 255).astype(np.uint8)
print(f"Brightness adjusted: shape={brighter.shape}, dtype={brighter.dtype}")

# 2. Threshold an entire image at once (binary mask)
mask = img > 128
print(f"Threshold mask: shape={mask.shape}, dtype={mask.dtype}")

# 3. Compute mean pixel value per channel, across the whole image
channel_means = img.mean(axis=(0, 1))  # shape (3,) — one mean per R,G,B
print(f"Per-channel means (R, G, B): {channel_means}")

# 4. Count how many pixels are above a threshold — no loop needed
bright_pixel_count = np.sum(img > 200)
total_pixels = img.size // 3  # total pixels (not elements)
print(f"Bright pixels (>200): {bright_pixel_count} / {total_pixels} ({bright_pixel_count/total_pixels*100:.1f}%)")

# 5. Normalize to [0, 1] range
normalized = img.astype(np.float32) / 255.0
print(f"Normalized range: [{normalized.min():.3f}, {normalized.max():.3f}]")

# 6. Standardize (zero mean, unit variance) per channel
img_f32 = img.astype(np.float32)
mean = img_f32.mean(axis=(0, 1))
std = img_f32.std(axis=(0, 1))
standardized = (img_f32 - mean) / std  # Broadcasting handles per-channel
print(f"Standardized per-channel mean: {standardized.mean(axis=(0,1))}")
print(f"Standardized per-channel std:  {standardized.std(axis=(0,1))}")

Brightness adjusted: shape=(480, 640, 3), dtype=uint8
Threshold mask: shape=(480, 640, 3), dtype=bool
Per-channel means (R, G, B): [127.40889974 127.68587891 127.45876628]
Bright pixels (>200): 198338 / 307200 (64.6%)
Normalized range: [0.000, 1.000]
Standardized per-channel mean: [-1.2984760e-05 -1.9440105e-05  4.0954226e-05]
Standardized per-channel std:  [0.99989915 0.9999851  0.9999728 ]


### More Loop vs Vectorised Benchmarks

Let's compare two more common Edge AI tasks to drive the point home. The pattern is always the same: **Python loop = slow, NumPy vectorised = fast**.

### Task 1: Add 20 to Every Pixel (Safely)

This is the brightness adjustment from the intermediate notebook. The loop version must:
- Read each pixel → convert to int → add 20 → clip to 255 → write back
- That's ~921,600 iterations of Python interpreter overhead

The vectorised version does it all in one compiled C call.

### Task 2: Compute Per-Channel Mean

The loop version sums all pixels per channel using nested loops. The vectorised version uses `img.mean(axis=(0, 1))` — one call, three means, done.

### The Key Takeaway

| Task | Loop approach | Vectorised approach | Typical speedup |
|---|---|---|---|
| Add value to all pixels | 3 nested for-loops | `np.clip(img.astype(int16) + val, 0, 255)` | ~100-500x |
| Per-channel mean | 2 nested for-loops per channel | `img.mean(axis=(0, 1))` | ~200-1000x |
| Normalize image | 3 nested for-loops | `img.astype(np.float32) / 255.0` | ~100-300x |

On a Raspberry Pi, the speedup is even more dramatic because the CPU is slower — every wasted Python interpreter cycle hurts more.

In [10]:
# --- Benchmark: more loop vs vectorised comparisons ---

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Task 1: Add 20 to every pixel (safely)
start = time.time()
result_loop = np.zeros_like(img)
for y in range(img.shape[0]):
    for x in range(img.shape[1]):
        for c in range(img.shape[2]):
            val = int(img[y, x, c]) + 20
            result_loop[y, x, c] = min(val, 255)
t1_loop = time.time() - start

start = time.time()
result_vec = np.clip(img.astype(np.int16) + 20, 0, 255).astype(np.uint8)
t1_vec = time.time() - start
print(f"Add 20 to every pixel:")
print(f"  Loop:      {t1_loop:.4f}s")
print(f"  Vectorised: {t1_vec:.4f}s  ({t1_loop/t1_vec:.1f}x faster)")

# Task 2: Compute per-channel mean
start = time.time()
means_loop = [0, 0, 0]
for c in range(3):
    total = 0
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            total += img[y, x, c]
    means_loop[c] = total / (img.shape[0] * img.shape[1])
t2_loop = time.time() - start

start = time.time()
means_vec = img.mean(axis=(0, 1))
t2_vec = time.time() - start
print(f"\nPer-channel mean:")
print(f"  Loop:      {t2_loop:.4f}s")
print(f"  Vectorised: {t2_vec:.4f}s  ({t2_loop/t2_vec:.1f}x faster)")
print(f"  Results match: {np.allclose(means_loop, means_vec)}")

Add 20 to every pixel:
  Loop:      0.3053s
  Vectorised: 0.0004s  (702.1x faster)


/var/folders/r0/q2bxbvrx3hbf3xsp0qjqrkq40000gn/T/ipykernel_64145/1855548262.py:29: RuntimeWarning: overflow encountered in scalar add
  total += img[y, x, c]



Per-channel mean:
  Loop:      0.2847s
  Vectorised: 0.0024s  (119.2x faster)
  Results match: False


---
## 4. Advanced Indexing Patterns

Beyond basic slicing, NumPy offers powerful indexing methods that enable complex data selection without loops.

### `np.meshgrid`: Coordinate Grids for Spatial Operations

`np.meshgrid` creates 2D coordinate grids from 1D ranges — essential for any operation that needs to know **where** each pixel is in the image (not just what its value is).

### What It Creates

For a 4×5 image, `meshgrid` generates two arrays:

```
Y coordinates (row index):     X coordinates (column index):
[[0 0 0 0 0]                   [[0 1 2 3 4]
 [1 1 1 1 1]                    [0 1 2 3 4]
 [2 2 2 2 2]                    [0 1 2 3 4]
 [3 3 3 3 3]]                   [0 1 2 3 4]]
```

Every pixel "knows" its own (y, x) position. This lets you compute spatial properties like distance, angle, or position-dependent effects.

### Common Use Cases

| Use case | How meshgrid helps |
|---|---|
| **Vignette** (darken edges) | Compute distance from center → multiply by distance |
| **Geometric transforms** | Map (x, y) → new coordinates for rotation, warping |
| **Distance fields** | `sqrt((y - cy)² + (x - cx)²)` for every pixel at once |
| **Gaussian filters** | Create 2D Gaussian kernels from 1D coordinate ranges |

### The `indexing` Parameter

- `indexing='ij'` → first array is rows (y), second is columns (x) — **use this for images**
- `indexing='xy'` (default) → first array is columns (x), second is rows (y) — MATLAB-style

For image processing, always use `indexing='ij'` to match the `(height, width)` convention.

In [11]:
# --- Advanced indexing: meshgrid for coordinate-based operations ---

# Create coordinate grids (useful for spatial operations on images)
height, width = 4, 5
y_coords, x_coords = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')

print("Y coordinates (row indices):")
print(y_coords)
print("\nX coordinates (column indices):")
print(x_coords)

# Use case: compute distance from center of an image
h, w = 480, 640
cy, cx = h // 2, w // 2
y, x = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
distances = np.sqrt((y - cy)**2 + (x - cx)**2)
print(f"\nDistance from center: shape={distances.shape}")
print(f"  Center distance: {distances[cy, cx]:.1f}")
print(f"  Corner distance: {distances[0, 0]:.1f}")

# Create a vignette mask (darker at edges)
max_dist = distances.max()
vignette = 1.0 - (distances / max_dist) * 0.5  # 1.0 at center, 0.5 at corners
print(f"  Vignette at center: {vignette[cy, cx]:.2f}")
print(f"  Vignette at corner: {vignette[0, 0]:.2f}")

Y coordinates (row indices):
[[0 0 0 0 0]
 [1 1 1 1 1]
 [2 2 2 2 2]
 [3 3 3 3 3]]

X coordinates (column indices):
[[0 1 2 3 4]
 [0 1 2 3 4]
 [0 1 2 3 4]
 [0 1 2 3 4]]

Distance from center: shape=(480, 640)
  Center distance: 0.0
  Corner distance: 400.0
  Vignette at center: 1.00
  Vignette at corner: 0.50


### Boolean Masks for Spatial Regions

Boolean masks aren't just for thresholding — you can create **spatial masks** (circles, rectangles, arbitrary shapes) to select regions of an image by position, not just by pixel value.

### Creating a Circular Mask

```python
y, x = np.ogrid[:h, :w]                    # coordinate grids
mask = (y - cy)**2 + (x - cx)**2 <= r**2   # circle equation
```

`np.ogrid` creates open grids (memory-efficient — only stores 1D arrays). The mask is `True` for all pixels inside the circle, `False` outside.

### Visual: Circular Mask on an 8×8 Image

```
Mask (radius=3, center at 4,4):
0 0 0 0 0 0 0 0
0 0 1 1 1 1 1 0
0 1 1 1 1 1 1 1
0 1 1 1 1 1 1 1
0 1 1 1 1 1 1 1
0 1 1 1 1 1 1 1
0 0 1 1 1 1 1 0
0 0 0 0 0 0 0 0
```

### Applying the Mask

```python
img[mask]              # 1D array of all pixels inside the circle
img[~mask] = 0         # black out everything outside the circle
img[mask].mean(axis=0) # average color of the circular region
```

The `~` operator inverts a boolean mask: `~mask` is `True` everywhere `mask` is `False`.

In [12]:
# --- Advanced indexing: selecting regions with boolean masks ---

img = np.random.randint(0, 256, (8, 8, 3), dtype=np.uint8)

# Create a circular mask (select pixels within a radius)
h, w = img.shape[:2]
cy, cx = h // 2, w // 2
radius = 3

y, x = np.ogrid[:h, :w]
circular_mask = (y - cy)**2 + (x - cx)**2 <= radius**2
print("Circular mask:")
print(circular_mask.astype(int))

# Apply the mask: set pixels outside the circle to black
masked_img = img.copy()
masked_img[~circular_mask] = 0  # ~ inverts the mask
print(f"\nMasked image shape: {masked_img.shape}")
print(f"Pixels inside circle: {circular_mask.sum()}")
print(f"Pixels outside circle: {(~circular_mask).sum()}")

# Select only the pixels inside the circle (flattened)
inside_pixels = img[circular_mask]  # 1D array of all pixels inside the circle
print(f"\nInside pixels: shape={inside_pixels.shape}, mean={inside_pixels.mean(axis=0)}")

Circular mask:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 1 1 1 1 1 0]
 [0 0 1 1 1 1 1 0]
 [0 1 1 1 1 1 1 1]
 [0 0 1 1 1 1 1 0]
 [0 0 1 1 1 1 1 0]
 [0 0 0 0 1 0 0 0]]

Masked image shape: (8, 8, 3)
Pixels inside circle: 29
Pixels outside circle: 35

Inside pixels: shape=(29, 3), mean=[131.44827586 131.96551724 131.44827586]


### `np.where` for Coordinate Extraction

`np.where(condition)` with only a condition argument returns the **coordinates** of all elements where the condition is `True` — one index array per dimension.

### Finding Pixel Locations

```python
y_coords, x_coords = np.where(img > 200)
# y_coords = [0, 3, 5, ...]  ← row indices of bright pixels
# x_coords = [4, 1, 2, ...]  ← column indices of bright pixels
```

This is the vectorized equivalent of scanning every pixel with a nested loop and collecting coordinates — but it runs in optimized C code.

### Use Cases in Edge AI

| Task | How `np.where` helps |
|---|---|
| Find bright spots (LEDs, headlights) | `np.where(img > 200)` gives their locations |
| Find edge pixels after edge detection | `np.where(edges > 0)` |
| Find pixels of a specific color | `np.where((img[:,:,0] > 200) & (img[:,:,2] < 50))` |
| Get bounding box of a region | Use min/max of the returned coordinates |

### `np.where` with Three Arguments

`np.where(condition, x, y)` is the vectorized if-else (covered in the intermediate notebook). With one argument, it's a coordinate finder. Same function, two different modes.

In [13]:
# --- np.where with multiple conditions ---

img = np.random.randint(0, 256, (6, 6), dtype=np.uint8)
print("Original image:")
print(img)

# Find coordinates of all pixels above a threshold
bright_y, bright_x = np.where(img > 200)
print(f"\nBright pixel coordinates (y, x) where img > 200:")
print(f"  Y coords: {bright_y}")
print(f"  X coords: {bright_x}")
print(f"  Count: {len(bright_y)}")

# np.where can also return values from two arrays based on a condition
positive = np.array([1, 2, 3, 4, 5])
negative = np.array([-1, -2, -3, -4, -5])
condition = np.array([True, False, True, False, True])
result = np.where(condition, positive, negative)
print(f"\nnp.where(condition, positive, negative): {result}")
# [1, -2, 3, -4, 5] — picks from positive where True, from negative where False

Original image:
[[202  49 172  40  10 136]
 [  5 163 111  65 194 139]
 [160  17  80 100 221 236]
 [ 82 150 221 106  34 252]
 [105 216  98 127  90 233]
 [  4  62 133 121 111 199]]

Bright pixel coordinates (y, x) where img > 200:
  Y coords: [0 2 2 3 3 4 4]
  X coords: [0 4 5 2 5 1 5]
  Count: 7

np.where(condition, positive, negative): [ 1 -2  3 -4  5]


---
## 5. Memory Layout: C-Order vs Fortran-Order

NumPy arrays can be stored in two different memory orders:

- **C-order (row-major)**: Default in NumPy. Last axis varies fastest. Rows are contiguous.
- **Fortran-order (column-major)**: First axis varies fastest. Columns are contiguous. Used by MATLAB, Fortran.

Understanding this helps when interfacing with libraries that expect a specific order, and for optimizing cache performance.

In [14]:
# --- C-order vs Fortran-order ---

# Default: C-order (row-major) — rows are contiguous in memory
arr_c = np.array([[1, 2, 3], [4, 5, 6]], order='C')
print("C-order (row-major):")
print(arr_c)
print(f"  C_CONTIGUOUS: {arr_c.flags['C_CONTIGUOUS']}")  # True
print(f"  F_CONTIGUOUS: {arr_c.flags['F_CONTIGUOUS']}")  # False
print(f"  Memory order: [1, 2, 3, 4, 5, 6] — row by row")

# Fortran-order (column-major) — columns are contiguous in memory
arr_f = np.array([[1, 2, 3], [4, 5, 6]], order='F')
print(f"\nFortran-order (column-major):")
print(arr_f)
print(f"  C_CONTIGUOUS: {arr_f.flags['C_CONTIGUOUS']}")  # False
print(f"  F_CONTIGUOUS: {arr_f.flags['F_CONTIGUOUS']}")  # True
print(f"  Memory order: [1, 4, 2, 5, 3, 6] — column by column")

# Both arrays look the same when printed — the difference is in memory layout
print(f"\nSame values? {np.array_equal(arr_c, arr_f)}")  # True

# Performance tip: accessing data in the order it's stored in memory is faster
# For C-order: iterate over the last axis (columns within a row)
# For Fortran-order: iterate over the first axis (rows within a column)

C-order (row-major):
[[1 2 3]
 [4 5 6]]
  C_CONTIGUOUS: True
  F_CONTIGUOUS: False
  Memory order: [1, 2, 3, 4, 5, 6] — row by row

Fortran-order (column-major):
[[1 2 3]
 [4 5 6]]
  C_CONTIGUOUS: False
  F_CONTIGUOUS: True
  Memory order: [1, 4, 2, 5, 3, 6] — column by column

Same values? True


### Cache Performance: Row Access vs Column Access

Memory layout isn't just about correctness — it directly affects **speed**. CPUs read memory in chunks (cache lines), so accessing data that's **next to each other in memory** is much faster than jumping around.

### The Cache Principle

```
C-order memory layout (row-major):
[0,0] [0,1] [0,2] ... [0,999] [1,0] [1,1] ... [1,999] ...

Row access (arr[i, :]):  reads contiguous memory → cache-friendly → FAST
Column access (arr[:, j]): reads scattered memory → cache misses → SLOW
```

### Why It Matters

When you access a row in C-order, the CPU loads a whole chunk of nearby values into its cache. The next few reads from that row are instant. But when you access a column, each element is far apart in memory — every read is a cache miss, forcing the CPU to fetch from main memory (10-100x slower).

### Practical Takeaway for Images

Since images are C-order (row-major) by default:
- **Process row-by-row** for best performance
- **Avoid column-by-column iteration** unless you transpose first
- If you must iterate over columns, consider `arr = np.asfortranarray(arr)` to switch to column-major order first

In [15]:
# --- Performance: row access vs column access ---

arr = np.random.rand(1000, 1000)

# Access by rows (C-order friendly — each row is contiguous)
start = time.time()
for i in range(1000):
    _ = arr[i, :]  # Row access — fast (contiguous in C-order)
row_time = time.time() - start

# Access by columns (C-order unfriendly — each column is non-contiguous)
start = time.time()
for j in range(1000):
    _ = arr[:, j]  # Column access — slower (strided in C-order)
col_time = time.time() - start

print(f"Row access (1000 rows):    {row_time:.4f}s")
print(f"Column access (1000 cols): {col_time:.4f}s")
print(f"Ratio: {col_time/row_time:.2f}x slower for column access in C-order")
print("\n→ For images (C-order), process row-by-row for best cache performance")

Row access (1000 rows):    0.0002s
Column access (1000 cols): 0.0002s
Ratio: 0.98x slower for column access in C-order

→ For images (C-order), process row-by-row for best cache performance


---
## 6. np.einsum() — Einstein Notation for Complex Operations

`np.einsum()` is a powerful function that can express many complex array operations compactly. It's especially useful for matrix operations in AI (attention mechanisms, batched matrix multiply, etc.).

### `np.einsum` Basics: Einstein Notation

`np.einsum` is a compact "mini-language" for expressing array operations. Instead of calling separate functions for matrix multiply, transpose, trace, diagonal, etc., you describe the operation with a **string subscript notation**.

### The Core Idea

You label each axis of each array with a letter, then specify which axes appear in the output:
- Axes that appear in **inputs but not output** → **summed over** (like a dot product)
- Axes that appear in **both input and output** → **kept** (like a matrix multiply)

### Common Patterns

| Operation | einsum string | Equivalent to |
|---|---|---|
| Matrix multiply | `'ij,jk->ik'` | `A @ B` |
| Transpose | `'ij->ji'` | `A.T` |
| Diagonal | `'ii->i'` | `np.diag(A)` |
| Trace (sum of diagonal) | `'ii->'` | `np.trace(A)` |
| Dot product | `'i,i->'` | `np.dot(a, b)` |
| Outer product | `'i,j->ij'` | `np.outer(a, b)` |
| Element-wise multiply | `'ij,ij->ij'` | `A * B` |

### Visual: Matrix Multiply `'ij,jk->ik'`

```
A: (i, j)    B: (j, k)    →    C: (i, k)
     ↑  ↑          ↑  ↑              ↑  ↑
     i  j          j  k              i  k

j appears in both inputs but NOT output → summed (dot product along j)
i and k appear in output → kept as dimensions of the result
```

### Why Use einsum?

- **One syntax for many operations** — no need to remember separate function names
- **Batched operations** — handles 3D+ arrays naturally (critical for deep learning)
- **Often more readable** for complex multi-array operations than chained `np.dot` calls

In [16]:
# --- np.einsum basics ---

# Matrix multiplication: C = A @ B
# einsum: 'ij,jk->ik' means: sum over j, result has indices i and k
A = np.random.rand(3, 4)
B = np.random.rand(4, 2)

# Standard matrix multiply
C_standard = A @ B

# einsum equivalent
C_einsum = np.einsum('ij,jk->ik', A, B)

print(f"A shape: {A.shape}, B shape: {B.shape}")
print(f"Standard A@B shape: {C_standard.shape}")
print(f"einsum shape: {C_einsum.shape}")
print(f"Results match: {np.allclose(C_standard, C_einsum)}")

# Transpose with einsum: 'ij->ji'
transposed = np.einsum('ij->ji', A)
print(f"\neinsum transpose: {np.allclose(transposed, A.T)}")

# Diagonal extraction: 'ii->i'
square = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
diag = np.einsum('ii->i', square)
print(f"Diagonal: {diag}")  # [1, 5, 9]

# Trace (sum of diagonal): 'ii->'
trace = np.einsum('ii->', square)
print(f"Trace: {trace}")  # 15

A shape: (3, 4), B shape: (4, 2)
Standard A@B shape: (3, 2)
einsum shape: (3, 2)
Results match: True

einsum transpose: True
Diagonal: [1 5 9]
Trace: 15


### `np.einsum` for Batched Operations (Deep Learning)

In deep learning, you constantly work with **batches** of matrices — e.g., attention mechanisms multiply batches of query and key matrices. `np.einsum` handles this elegantly.

### Batched Matrix Multiply

```
Standard matmul:  A @ B     → einsum: 'ij,jk->ik'
Batched matmul:   A[b] @ B[b] for each b → einsum: 'bmk,bkn->bmn'
```

The `b` index is the batch dimension — einsum processes all batches in **one call**, no Python loop needed.

### Breaking Down `'bmk,bkn->bmn'`

```
Input A: (b, m, k)    Input B: (b, k, n)
         ↑  ↑  ↑              ↑  ↑  ↑
         b  m  k              b  k  n

Output:  (b, m, n)  — b and m kept, k summed (appears in inputs but not output)
```

- `b` appears in both inputs and output → **batch dimension** (kept as-is)
- `m` appears in input A and output → **row dimension** of result
- `n` appears in input B and output → **column dimension** of result
- `k` appears in both inputs but NOT output → **summed over** (this is the dot-product axis)

### Batched Dot Product: `'bi,bi->b'`

```
A: (batch, features)    B: (batch, features)
         ↑  ↑                    ↑  ↑
         b  i                    b  i

Output: (batch,)  — i is summed, b is kept → one dot product per batch element
```

This computes the dot product of corresponding rows across a batch — common in similarity scoring and attention.

In [17]:
# --- einsum for batched operations (common in deep learning) ---

# Batched matrix multiply: (batch, m, k) @ (batch, k, n) → (batch, m, n)
batch_size = 4
A_batch = np.random.rand(batch_size, 3, 4)  # 4 matrices of 3x4
B_batch = np.random.rand(batch_size, 4, 2)  # 4 matrices of 4x2

# einsum for batched matmul: 'bmk,bkn->bmn'
C_batch = np.einsum('bmk,bkn->bmn', A_batch, B_batch)
print(f"Batched matmul: {A_batch.shape} @ {B_batch.shape} → {C_batch.shape}")
# (4, 3, 4) @ (4, 4, 2) → (4, 3, 2)

# Verify with a loop
C_loop = np.zeros((batch_size, 3, 2))
for b in range(batch_size):
    C_loop[b] = A_batch[b] @ B_batch[b]
print(f"Matches loop version: {np.allclose(C_batch, C_loop)}")

# Dot product of corresponding rows: 'bi,bi->b'
a = np.random.rand(4, 3)
b = np.random.rand(4, 3)
dots = np.einsum('bi,bi->b', a, b)
print(f"\nBatched dot product: shape={dots.shape}")
print(f"  Matches manual: {np.allclose(dots, (a * b).sum(axis=1))}")

Batched matmul: (4, 3, 4) @ (4, 4, 2) → (4, 3, 2)
Matches loop version: True

Batched dot product: shape=(4,)
  Matches manual: True


---
## 7. Performance Optimization Techniques

### Key strategies for Edge AI:

1. **Avoid loops** — use vectorised operations
2. **Use the right dtype** — `float32` instead of `float64` saves 50% memory
3. **Pre-allocate output arrays** — avoid repeated allocations
4. **Use `np.save`/`np.load`** for efficient binary storage (faster than CSV)
5. **Avoid unnecessary copies** — use views when safe

### Pre-Allocation vs Dynamic Growth

When building up arrays incrementally (e.g., collecting results from multiple camera frames), how you accumulate data has a big performance impact.

### The Problem with Appending

```python
results = []
for i in range(10000):
    results.append(np.random.rand(3))
results = np.array(results)
```

Every `append` may trigger a **memory reallocation** — Python lists grow by over-allocating, but converting the list to a NumPy array at the end still requires a full copy. This is wasteful.

### The Fix: Pre-Allocate

If you know the final size, allocate the full array upfront and fill it:

```python
results = np.empty((10000, 3))   # allocate once
for i in range(10000):
    results[i] = np.random.rand(3)
```

`np.empty` creates an uninitialized array — it doesn't waste time filling with zeros (unlike `np.zeros`). You immediately overwrite every element, so the initial garbage values don't matter.

### Three Levels of Performance

| Approach | Why it's fast/slow |
|---|---|
| Append + convert | Multiple reallocations + final copy |
| Pre-allocate + fill loop | One allocation, but still has Python loop overhead |
| Fully vectorised | One allocation, no loop — NumPy handles everything in C |

In [18]:
# --- Pre-allocation vs dynamic growth ---

# SLOW: appending to a list and converting (causes reallocations)
start = time.time()
results = []
for i in range(10000):
    results.append(np.random.rand(3))
results = np.array(results)
t_append = time.time() - start
print(f"Append + convert: {t_append:.4f}s, shape={results.shape}")

# FAST: pre-allocate the array and fill it
start = time.time()
results_pre = np.empty((10000, 3))  # Pre-allocate
for i in range(10000):
    results_pre[i] = np.random.rand(3)
t_prealloc = time.time() - start
print(f"Pre-allocated:     {t_prealloc:.4f}s, shape={results_pre.shape}")

# FASTEST: fully vectorised (no loop at all)
start = time.time()
results_vec = np.random.rand(10000, 3)
t_vec = time.time() - start
print(f"Fully vectorised:  {t_vec:.4f}s, shape={results_vec.shape}")

print(f"\n→ Pre-allocation is {t_append/t_prealloc:.1f}x faster than appending")
print(f"→ Vectorised is {t_append/t_vec:.1f}x faster than appending")

Append + convert: 0.0082s, shape=(10000, 3)
Pre-allocated:     0.0050s, shape=(10000, 3)
Fully vectorised:  0.0001s, shape=(10000, 3)

→ Pre-allocation is 1.6x faster than appending
→ Vectorised is 87.5x faster than appending


### Memory-Efficient Storage: `np.save` / `np.load`

When saving arrays to disk, the format you choose has a huge impact on file size and load speed.

### `.npy` vs `.csv`

| Format | How it stores data | Size for 10×10 array | Preserves dtype? | Load speed |
|---|---|---|---|---|
| `.npy` (binary) | Raw bytes — exactly what's in memory | ~880 bytes | Yes (uint8 stays uint8) | Fast |
| `.csv` (text) | Human-readable numbers as text | ~2,000+ bytes | No (everything becomes float) | Slow (parsing) |

### Why `.npy` Is Better for Edge AI

- **Smaller files**: binary stores numbers in their native encoding, not as text characters
- **Faster loading**: no text parsing needed — just read bytes directly into memory
- **Preserves dtype**: a `uint8` array saves as `uint8` and loads back as `uint8` — no surprises
- **Preserves shape**: a 3D array saves and loads with its exact shape intact

### When to Use CSV Instead

CSV is useful when you need **human readability** (debugging, sharing with non-programmers) or **interoperability** with tools that only accept text formats. For everything else in Edge AI, use `.npy`.

In [19]:
# --- Memory-efficient storage with np.save/np.load ---

import os

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Save as .npy (NumPy binary format) — compact and fast
np.save('/tmp/test_image.npy', img)
npy_size = os.path.getsize('/tmp/test_image.npy')
print(f"npy file size: {npy_size:,} bytes")

# Load it back — preserves dtype and shape exactly
loaded = np.load('/tmp/test_image.npy')
print(f"Loaded shape: {loaded.shape}, dtype: {loaded.dtype}")
print(f"Data matches: {np.array_equal(img, loaded)}")

# Compare with CSV (text format) — much larger and slower
# (We won't actually save CSV for a 480x640x3 image — it would be huge)
# But for a small array:
small = np.random.rand(10, 10)
np.save('/tmp/small.npy', small)
np.savetxt('/tmp/small.csv', small, delimiter=',')
print(f"\nSmall 10x10 array:")
print(f"  npy: {os.path.getsize('/tmp/small.npy')} bytes")
print(f"  csv: {os.path.getsize('/tmp/small.csv')} bytes")
print(f"  → npy is {os.path.getsize('/tmp/small.csv') / os.path.getsize('/tmp/small.npy'):.1f}x smaller")

npy file size: 921,728 bytes
Loaded shape: (480, 640, 3), dtype: uint8
Data matches: True

Small 10x10 array:
  npy: 928 bytes
  csv: 2500 bytes
  → npy is 2.7x smaller


### Avoiding Unnecessary Memory Copies

Every intermediate array in a computation chain is a **separate memory allocation**. On memory-constrained devices, these copies add up fast — both in memory usage and in time spent allocating and copying.

### The Problem: Chained Copies

```python
step1 = img.astype(np.float32)    # copy 1: new array for float conversion
step2 = step1 / 255.0             # copy 2: new array for division result
step3 = step2 - 0.5               # copy 3: new array for subtraction
step4 = step3 * 2.0               # copy 4: new array for multiplication
```

That's **4 separate memory allocations** for a single normalization pipeline. On a 480×640×3 image, that's ~14 MB of temporary memory.

### Three Levels of Optimization

| Approach | Memory copies | How it works |
|---|---|---|
| **Chained separate steps** | 4 copies | Each step creates a new array |
| **Single expression** | 1-2 copies | NumPy may optimize the chain internally |
| **In-place operations** | 1 copy | `*=`, `+=`, `-=`, `/=` modify the array directly |

### In-Place Operations

```python
result = img.astype(np.float32)   # 1 copy (unavoidable — dtype change)
result /= 255.0                    # in-place — no new array
result -= 0.5                      # in-place — no new array
result *= 2.0                      # in-place — no new array
```

⚠️ In-place operations modify the original array. Only use them when you don't need the original data anymore, or when you've already made a copy (like the `astype` above).

In [20]:
# --- Avoiding unnecessary copies ---

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# BAD: creates unnecessary intermediate copies
start = time.time()
step1 = img.astype(np.float32)           # copy 1
step2 = step1 / 255.0                     # copy 2
step3 = step2 - 0.5                       # copy 3
step4 = step3 * 2.0                       # copy 4
t_copies = time.time() - start
print(f"Multiple intermediate copies: {t_copies:.4f}s")

# GOOD: chain operations (NumPy optimizes the chain)
start = time.time()
result = (img.astype(np.float32) / 255.0 - 0.5) * 2.0
t_chained = time.time() - start
print(f"Chained operations:           {t_chained:.4f}s")

# BEST: use in-place operations where possible
start = time.time()
result_inplace = img.astype(np.float32)  # one copy
result_inplace /= 255.0                   # in-place
result_inplace -= 0.5                     # in-place
result_inplace *= 2.0                     # in-place
t_inplace = time.time() - start
print(f"In-place operations:          {t_inplace:.4f}s")

print(f"\nAll produce same result: {np.allclose(step4, result) and np.allclose(result, result_inplace)}")
print("\n→ In-place ops (*=, +=, -=, /=) modify the array without creating copies")

Multiple intermediate copies: 0.0012s
Chained operations:           0.0010s
In-place operations:          0.0005s

All produce same result: True

→ In-place ops (*=, +=, -=, /=) modify the array without creating copies


### Profiling Your NumPy Code

When optimizing for edge devices, **measure, don't guess**. A simple timing wrapper can tell you which approach is actually faster on your specific hardware.

### The Benchmarking Pattern

```python
def benchmark(func, *args, repeat=5, **kwargs):
    times = []
    for _ in range(repeat):
        start = time.time()
        result = func(*args, **kwargs)
        times.append(time.time() - start)
    return sum(times) / len(times), result
```

### Micro-Optimizations That Matter on Edge Devices

| Optimization | Why it helps | Example |
|---|---|---|
| Multiply instead of divide | Division is ~2-3x slower than multiplication on most CPUs | `x * (1/255)` instead of `x / 255` |
| Use `float32` not `float64` | Half the memory, faster cache utilization, most models don't need double precision | `dtype=np.float32` |
| Pre-compute constants | Avoid recalculating the same value every frame | `INV_255 = 1.0 / 255.0` stored once |
| Batch operations | One call to process N items is faster than N calls processing 1 item | `np.stack` images into a batch |

### Why Repetition Matters

A single run can be misleading — the OS might have been busy, or the CPU cache was cold. Running 5+ times and averaging gives a more reliable measurement.

In [21]:
# --- Summary: profiling your NumPy code ---

# A simple profiling helper for Edge AI code
def benchmark(func, *args, repeat=5, **kwargs):
    """Run a function multiple times and return the average time."""
    times = []
    for _ in range(repeat):
        start = time.time()
        result = func(*args, **kwargs)
        times.append(time.time() - start)
    avg = sum(times) / len(times)
    return avg, result

# Example: compare different normalization approaches
img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

def normalize_divide(arr):
    return arr.astype(np.float32) / 255.0

def normalize_multiply(arr):
    return arr.astype(np.float32) * (1.0 / 255.0)

t1, _ = benchmark(normalize_divide, img)
t2, _ = benchmark(normalize_multiply, img)

print(f"Normalization by division:  {t1:.6f}s avg")
print(f"Normalization by multiply:  {t2:.6f}s avg")
print(f"\n→ Multiplication is often slightly faster than division")
print(f"→ On a Pi, these micro-optimizations add up across thousands of frames")

Normalization by division:  0.000312s avg
Normalization by multiply:  0.000355s avg

→ Multiplication is often slightly faster than division
→ On a Pi, these micro-optimizations add up across thousands of frames


---
## Quick Recap

1. **`np.frombuffer()`** avoids copying raw camera data — it wraps existing memory as an array. The result is **read-only by default**; use `.copy()` if you need to modify it.
2. **`np.ascontiguousarray()`** fixes memory layout issues after transposes/fancy slicing. Always check `C_CONTIGUOUS` before passing arrays to external libraries.
3. **Vectorised NumPy operations** can be ~100x faster than Python loops — because they use compiled, NEON-accelerated code instead of the slow Python interpreter.
4. **Pre-allocate arrays** instead of appending to lists — avoids costly reallocations.
5. **In-place operations** (`*=`, `+=`, `-=`, `/=`) avoid unnecessary memory copies.
6. **`np.einsum()`** provides a compact notation for complex matrix operations, including batched operations common in deep learning.
7. **C-order (row-major)** is NumPy's default — access data row-by-row for best cache performance.

## Self-Check Questions

1. Why is the array returned by `np.frombuffer()` read-only by default, and how would you get a writable version?
2. When would you need to call `np.ascontiguousarray()` on an array, and what problem does skipping it cause?
3. Write two versions of code that adds 20 to every pixel value in an image (safely, without wraparound) — one using a Python loop, one fully vectorised — and explain which one you'd actually use in a real project, and why.
4. What's the difference between C-order and Fortran-order memory layout? Which does NumPy use by default?
5. Explain what `np.einsum('bmk,bkn->bmn', A, B)` does. What would the equivalent loop look like?